# Second-Order Methods (Solution Notebook)

This notebook contains simple reference implementations of:

- Newton's method
- BFGS
- Gauss--Newton

The goal is to keep the code as simple as possible for classroom use.

For each method, we report:
- the evolution of the cost function,
- the number of iterations,
- the final solution.


## 1. Imports

We use:
- `numpy` for vector and matrix operations,
- `matplotlib` to plot the cost history.


In [ ]:
# Import NumPy for numerical computations.
import numpy as np

# Import matplotlib to plot the cost history.
import matplotlib.pyplot as plt

## 2. Newton's Method and BFGS: Test Function

For Newton's method and BFGS, we use the smooth function

\begin{equation}
f(x,y)=\frac{1}{2}(x-1)^2 + (y+2)^2 + 0.1(x-1)^4.
\end{equation}

Its gradient is

\begin{equation}
\nabla f(x,y)=
\begin{pmatrix}
(x-1)+0.4(x-1)^3 \\
2(y+2)
\end{pmatrix},
\end{equation}

and its Hessian is

\begin{equation}
\nabla^2 f(x,y)=
\begin{pmatrix}
1+1.2(x-1)^2 & 0 \\
0 & 2
\end{pmatrix}.
\end{equation}

The minimizer is $ (x^\star,y^\star)=(1,-2) $.


In [ ]:
# Define the objective function.
def f(theta):
    # Extract the first variable.
    x = theta[0]
    # Extract the second variable.
    y = theta[1]
    # Return the function value.
    return 0.5 * (x - 1)**2 + (y + 2)**2 + 0.1 * (x - 1)**4

# Define the gradient of the objective function.
def grad_f(theta):
    # Extract the first variable.
    x = theta[0]
    # Extract the second variable.
    y = theta[1]
    # Return the gradient vector.
    return np.array([
        (x - 1) + 0.4 * (x - 1)**3,
        2 * (y + 2)
    ])

# Define the Hessian matrix of the objective function.
def hess_f(theta):
    # Extract the first variable.
    x = theta[0]
    # Return the Hessian matrix.
    return np.array([
        [1 + 1.2 * (x - 1)**2, 0.0],
        [0.0, 2.0]
    ])

## 3. Newton's Method

Newton's method computes the step $ \mathbf{p}_k $ from

\begin{equation}
\nabla^2 f(\mathbf{x}_k)\mathbf{p}_k = -\nabla f(\mathbf{x}_k),
\end{equation}

and then updates

\begin{equation}
\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha_k \mathbf{p}_k.
\end{equation}

To keep the method robust, we use a simple Armijo backtracking line search.


In [ ]:
# Define Newton's method with Armijo backtracking.
def newton_method(theta0, alpha0=1.0, rho=0.5, c=1e-4, tol=1e-8, max_iter=100):
    # Copy the initial point.
    theta = theta0.astype(float).copy()
    # Create a list to store the costs.
    costs = []

    # Start the main iteration loop.
    for k in range(max_iter):
        # Compute the current cost.
        cost = f(theta)
        # Store the current cost.
        costs.append(cost)
        # Compute the current gradient.
        g = grad_f(theta)

        # Stop if the gradient norm is small.
        if np.linalg.norm(g) < tol:
            # Exit the loop.
            break

        # Compute the current Hessian matrix.
        H = hess_f(theta)
        # Solve the Newton system H p = -g.
        p = np.linalg.solve(H, -g)

        # Initialize the trial step size.
        alpha = alpha0

        # Apply Armijo backtracking.
        while f(theta + alpha * p) > cost + c * alpha * np.dot(g, p):
            # Reduce the step size.
            alpha = rho * alpha

        # Update the iterate.
        theta = theta + alpha * p

    # Return the final point, the cost history, and the number of iterations.
    return theta, costs, k + 1

## 4. BFGS

BFGS is a quasi-Newton method. It builds an approximation $ H_k $ of the inverse Hessian and uses the direction

\begin{equation}
\mathbf{p}_k = -H_k \nabla f(\mathbf{x}_k).
\end{equation}

Then it updates the inverse Hessian approximation using the step

\begin{equation}
\mathbf{s}_k = \mathbf{x}_{k+1} - \mathbf{x}_k
\end{equation}

and the gradient difference

\begin{equation}
\mathbf{y}_k = \nabla f(\mathbf{x}_{k+1}) - \nabla f(\mathbf{x}_k).
\end{equation}

Again, we use a simple Armijo line search.


In [ ]:
# Define the BFGS method with Armijo backtracking.
def bfgs_method(theta0, alpha0=1.0, rho=0.5, c=1e-4, tol=1e-8, max_iter=100):
    # Copy the initial point.
    theta = theta0.astype(float).copy()
    # Compute the dimension of the problem.
    n = len(theta)
    # Initialize the inverse Hessian approximation with the identity.
    H = np.eye(n)
    # Create a list to store the costs.
    costs = []

    # Start the main iteration loop.
    for k in range(max_iter):
        # Compute the current cost.
        cost = f(theta)
        # Store the current cost.
        costs.append(cost)
        # Compute the current gradient.
        g = grad_f(theta)

        # Stop if the gradient norm is small.
        if np.linalg.norm(g) < tol:
            # Exit the loop.
            break

        # Compute the BFGS search direction.
        p = -H @ g

        # Initialize the trial step size.
        alpha = alpha0

        # Apply Armijo backtracking.
        while f(theta + alpha * p) > cost + c * alpha * np.dot(g, p):
            # Reduce the step size.
            alpha = rho * alpha

        # Compute the new point.
        theta_new = theta + alpha * p
        # Compute the new gradient.
        g_new = grad_f(theta_new)

        # Compute the step difference.
        s = theta_new - theta
        # Compute the gradient difference.
        y = g_new - g

        # Compute the curvature quantity.
        ys = np.dot(y, s)

        # Update the inverse Hessian approximation only if the curvature is positive.
        if ys > 1e-12:
            # Compute the scaling factor.
            r = 1.0 / ys
            # Compute the identity matrix.
            I = np.eye(n)
            # Apply the BFGS inverse update.
            H = (I - r * np.outer(s, y)) @ H @ (I - r * np.outer(y, s)) + r * np.outer(s, s)

        # Accept the new point.
        theta = theta_new

    # Return the final point, the cost history, and the number of iterations.
    return theta, costs, k + 1

## 5. Gauss--Newton: Nonlinear Least-Squares Problem

For Gauss--Newton, we solve a nonlinear least-squares problem.

We fit the model

\begin{equation}
\hat{y}(t;a,b)=a e^{bt}
\end{equation}

to data $ (t_i,y_i) $. The residuals are

\begin{equation}
r_i(a,b)=a e^{b t_i}-y_i.
\end{equation}

The least-squares cost is

\begin{equation}
F(a,b)=\frac{1}{2}\sum_{i=1}^{m} r_i(a,b)^2.
\end{equation}

Gauss--Newton computes the step from

\begin{equation}
(J^T J)\mathbf{p} = -J^T \mathbf{r},
\end{equation}

where $ J $ is the Jacobian of the residual vector.


In [ ]:
# Define a small synthetic dataset.
t_data = np.array([0.0, 0.5, 1.0, 1.5, 2.0])

# Define the observed values.
y_data = np.array([2.0, 2.6, 3.6, 4.9, 6.8])

# Define the residual vector.
def residual(theta):
    # Extract parameter a.
    a = theta[0]
    # Extract parameter b.
    b = theta[1]
    # Return the residual vector.
    return a * np.exp(b * t_data) - y_data

# Define the least-squares cost.
def F(theta):
    # Compute the residual vector.
    r = residual(theta)
    # Return one half of the squared norm.
    return 0.5 * np.dot(r, r)

# Define the Jacobian of the residual vector.
def jacobian(theta):
    # Extract parameter a.
    a = theta[0]
    # Extract parameter b.
    b = theta[1]
    # Compute the first Jacobian column.
    col1 = np.exp(b * t_data)
    # Compute the second Jacobian column.
    col2 = a * t_data * np.exp(b * t_data)
    # Return the Jacobian matrix.
    return np.column_stack([col1, col2])

In [ ]:
# Define the Gauss--Newton method with simple backtracking.
def gauss_newton(theta0, alpha0=1.0, rho=0.5, c=1e-4, tol=1e-8, max_iter=100):
    # Copy the initial point.
    theta = theta0.astype(float).copy()
    # Create a list to store the costs.
    costs = []

    # Start the main iteration loop.
    for k in range(max_iter):
        # Compute the current cost.
        cost = F(theta)
        # Store the current cost.
        costs.append(cost)
        # Compute the current residual vector.
        r = residual(theta)
        # Compute the current Jacobian matrix.
        J = jacobian(theta)
        # Compute the gradient J^T r.
        g = J.T @ r

        # Stop if the gradient norm is small.
        if np.linalg.norm(g) < tol:
            # Exit the loop.
            break

        # Form the Gauss--Newton matrix J^T J.
        A = J.T @ J
        # Solve the Gauss--Newton system.
        p = np.linalg.solve(A, -g)

        # Initialize the trial step size.
        alpha = alpha0

        # Apply simple backtracking.
        while F(theta + alpha * p) > cost + c * alpha * np.dot(g, p):
            # Reduce the step size.
            alpha = rho * alpha

        # Update the iterate.
        theta = theta + alpha * p

    # Return the final point, the cost history, and the number of iterations.
    return theta, costs, k + 1

## 6. Run the Methods

We now run:
- Newton's method,
- BFGS,
- Gauss--Newton.


In [ ]:
# Define the initial point for Newton and BFGS.
theta0_second_order = np.array([4.0, 3.0])

# Run Newton's method.
theta_newton, costs_newton, it_newton = newton_method(theta0_second_order)

# Run BFGS.
theta_bfgs, costs_bfgs, it_bfgs = bfgs_method(theta0_second_order)

# Define the initial point for Gauss--Newton.
theta0_gn = np.array([1.0, 0.1])

# Run Gauss--Newton.
theta_gn, costs_gn, it_gn = gauss_newton(theta0_gn)

# Print the Newton result.
print("Newton's method")
print("final solution =", theta_newton)
print("iterations     =", it_newton)
print()

# Print the BFGS result.
print("BFGS")
print("final solution =", theta_bfgs)
print("iterations     =", it_bfgs)
print()

# Print the Gauss--Newton result.
print("Gauss--Newton")
print("final solution =", theta_gn)
print("iterations     =", it_gn)

In [ ]:
# Create a figure.
plt.figure(figsize=(8, 5))

# Plot the cost history of Newton's method.
plt.plot(costs_newton, label="Newton")

# Plot the cost history of BFGS.
plt.plot(costs_bfgs, label="BFGS")

# Plot the cost history of Gauss--Newton.
plt.plot(costs_gn, label="Gauss--Newton")

# Label the horizontal axis.
plt.xlabel("iteration")

# Label the vertical axis.
plt.ylabel("cost")

# Add a title.
plt.title("Cost evolution")

# Add a legend.
plt.legend()

# Add a grid.
plt.grid(True)

# Show the figure.
plt.show()

## 7. Comparison with Standard Libraries

We compare our simple implementations with standard library calls:

- `scipy.optimize.minimize(..., method="BFGS")` for BFGS,
- `scipy.optimize.minimize(..., method="Newton-CG")` for a Newton-type method,
- `scipy.optimize.least_squares(..., method="lm")` for a Gauss--Newton / Levenberg--Marquardt type least-squares solver.


In [ ]:
# Import the SciPy optimizers.
from scipy.optimize import minimize, least_squares

In [ ]:
# Call SciPy BFGS on the smooth objective.
result_bfgs = minimize(f, theta0_second_order, jac=grad_f, method="BFGS")

# Print the SciPy BFGS result.
print("SciPy BFGS")
print("final solution =", result_bfgs.x)
print("iterations     =", result_bfgs.nit)
print("final cost     =", result_bfgs.fun)
print()

# Call SciPy Newton-CG on the smooth objective.
result_newton = minimize(f, theta0_second_order, jac=grad_f, hess=hess_f, method="Newton-CG")

# Print the SciPy Newton-CG result.
print("SciPy Newton-CG")
print("final solution =", result_newton.x)
print("iterations     =", result_newton.nit)
print("final cost     =", result_newton.fun)
print()

# Call SciPy least_squares on the nonlinear least-squares problem.
result_gn = least_squares(residual, theta0_gn, jac=jacobian, method="lm")

# Print the SciPy least-squares result.
print("SciPy least_squares (lm)")
print("final solution =", result_gn.x)
print("function evals =", result_gn.nfev)
print("final cost     =", 0.5 * np.dot(result_gn.fun, result_gn.fun))